# LightGBM Inference on MMELON Embeddings

Score a test set with the fold models from [MMELON_embedding_lgbm_cv.ipynb](MMELON_embedding_lgbm_cv.ipynb).
Each molecule is scored by every fold model, and the final `score` is their average (an ensemble).

**Two steps**
1. Extract embeddings for the test CSV with [MMELON_embedding_extraction.ipynb](MMELON_embedding_extraction.ipynb),
   using the **same embedding model** (base or fine-tuned) as the LightGBM run was trained on.
2. Run this notebook with that embedding folder and the LightGBM run folder.

**Environment:** run with the `biomed-multiview` kernel.

**Inputs**
- The LightGBM run folder (`output/<DATASET>/lgbm/<RUN_NAME>/`). The view (text, image, graph or fused) is read from it.
- The test embedding folder (`output/<DATASET>/embeddings/<EMBEDDING_NAME>/`).
- Optionally the test file (CSV or parquet) the embeddings came from: its SMILES, label and `KEEP_COLS`
  columns are kept in the output, and if it has a
  0/1 label column, metrics are reported. Without it, the output has only `row_index` and `smiles`.

**Metrics** (only with labels), for each fold model and for the ensemble
- **hit@K** (hit@200, hit@100, hit@50): rank molecules by score; hit@K is how many of the top K have label 1.
- AUROC, PR AUC, and precision, recall, specificity and accuracy at `THRESHOLD`.

**Output**, in `output/<DATASET>/lgbm/<RUN_NAME>/predictions/<PREDICTION_NAME>/`:
```
predictions.csv   # every row, plus score_fold0, score_fold1, ..., score (mean over folds),
                  # pred and rank (1 = highest score); rows without an embedding (invalid SMILES) have empty scores
metrics.csv       # one row per fold model plus the ensemble; only if labels are available
config.json       # settings used
```

## 1. Inputs and settings

**Edit these before running.** Data files can be `.csv` or `.parquet`; only the columns the notebook needs are read.

In [1]:
from pathlib import Path

# Dataset name: all outputs go under aircheck/output/<DATASET>/ (the pretrained base model is shared)
DATASET = "sample-data"

# Name of this prediction; outputs go to <LGBM run folder>/predictions/<PREDICTION_NAME>/
PREDICTION_NAME = "test"

# LightGBM run from MMELON_embedding_lgbm_cv.ipynb (contains models/fold_0.txt, ...)
LGBM_RUN_DIR = Path("../output") / DATASET / "lgbm" / "lgbm-fused-finetuned"

# Test embeddings from MMELON_embedding_extraction.ipynb
EMBEDDING_DIR = Path("../output") / DATASET / "embeddings" / "test-finetuned"

# The test CSV the embeddings came from (set to None to output only row_index and smiles)
DATA_PATH = Path("~/repos/biomed-multi-view/aircheck/sample_data/test.parquet")
SMILES_COL = "SMILES"
LABEL_COL = "LABEL"
KEEP_COLS = []        # other input columns to copy into predictions.csv, e.g. ["ID"]

In [2]:
THRESHOLD = 0.5           # score above this -> pred = 1
HIT_KS = [200, 100, 50]   # hit@K = number of label-1 molecules among the K highest-scored

In [3]:
import json

LGBM_RUN_DIR = Path(LGBM_RUN_DIR).expanduser().resolve()
EMBEDDING_DIR = Path(EMBEDDING_DIR).expanduser().resolve()
assert (LGBM_RUN_DIR / "config.json").exists(), f"Not a LightGBM run folder: {LGBM_RUN_DIR}"
assert (EMBEDDING_DIR / "config.json").exists(), f"Not an embedding folder: {EMBEDDING_DIR}"

lgbm_config = json.loads((LGBM_RUN_DIR / "config.json").read_text())
embedding_config = json.loads((EMBEDDING_DIR / "config.json").read_text())
VIEW = lgbm_config["view"]

MODEL_PATHS = sorted((LGBM_RUN_DIR / "models").glob("fold_*.txt"), key=lambda p: int(p.stem.split("_")[1]))
assert MODEL_PATHS, f"No models/fold_*.txt in {LGBM_RUN_DIR}"

# The test embeddings must match what the LightGBM models were trained on
assert (EMBEDDING_DIR / VIEW).is_dir(), (
    f"The LightGBM run uses {VIEW!r} embeddings, but {EMBEDDING_DIR} has only "
    f"{[p.name for p in EMBEDDING_DIR.iterdir() if p.is_dir()]}"
)
assert embedding_config["model"] == lgbm_config["embedding_model"], (
    f"Embedding model mismatch: LightGBM was trained on embeddings from {lgbm_config['embedding_model']!r}, "
    f"but the test embeddings come from {embedding_config['model']!r}"
)
if DATA_PATH is not None:
    assert Path(DATA_PATH).exists(), f"Data not found: {DATA_PATH}"
    if Path(embedding_config["data_path"]) != Path(DATA_PATH).resolve():
        print(f"WARNING: embeddings were extracted from {embedding_config['data_path']},\n"
              f"         not {Path(DATA_PATH).resolve()}. SMILES are checked row by row below.")

OUT_DIR = LGBM_RUN_DIR / "predictions" / PREDICTION_NAME
assert not (OUT_DIR.exists() and any(OUT_DIR.iterdir())), (
    f"{OUT_DIR} already exists. Choose a new PREDICTION_NAME or delete that folder."
)
OUT_DIR.mkdir(parents=True, exist_ok=True)
config = dict(
    lgbm_run_dir=str(LGBM_RUN_DIR), models=[str(p) for p in MODEL_PATHS], view=VIEW,
    embedding_dir=str(EMBEDDING_DIR), embedding_model=embedding_config["model"],
    data_path=str(DATA_PATH) if DATA_PATH else None, smiles_col=SMILES_COL, label_col=LABEL_COL,
    threshold=THRESHOLD, hit_ks=HIT_KS,
)
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=2))

print(f"LightGBM   : {len(MODEL_PATHS)} fold models on {VIEW} embeddings ({LGBM_RUN_DIR.name})")
print(f"Embeddings : {EMBEDDING_DIR / VIEW} (model: {embedding_config['model']})")
print(f"Data       : {DATA_PATH}")
print(f"Output     : {OUT_DIR}")

LightGBM   : 1 fold models on fused embeddings (lgbm-fused-finetuned)
Embeddings : ~/repos/biomed-multi-view/aircheck/output/sample-data/embeddings/test-finetuned/fused (model: ~/repos/biomed-multi-view/aircheck/output/sample-data/finetuned/mmelon-final_20261020_162634/best_model.ckpt)
Data       : ~/repos/biomed-multi-view/aircheck/sample_data/test.parquet
Output     : ~/repos/biomed-multi-view/aircheck/output/sample-data/lgbm/lgbm-fused-finetuned/predictions/test


## 2. Load the test embeddings

In [4]:
import numpy as np
import pandas as pd

import pyarrow.parquet as pq


def read_table(path, columns, optional=()) -> pd.DataFrame:
    # Read only the needed columns from a .csv or .parquet file; optional columns are read if present
    path = Path(path)
    is_parquet = path.suffix.lower() in (".parquet", ".pq")
    available = pq.ParquetFile(path).schema_arrow.names if is_parquet else pd.read_csv(path, nrows=0).columns.tolist()
    missing = [c for c in columns if c not in available]
    assert not missing, f"{path.name} has no column(s) {missing}; available: {available}"
    wanted = list(dict.fromkeys([*columns, *(c for c in optional if c in available)]))
    df = pd.read_parquet(path, columns=wanted) if is_parquet else pd.read_csv(path, usecols=wanted)
    return df[wanted].reset_index(drop=True)


emb = pd.read_parquet(EMBEDDING_DIR / VIEW).sort_values("row_index").set_index("row_index")
X = np.stack(emb["embedding"].to_numpy()).astype(np.float32)

if DATA_PATH is not None:
    rows = read_table(DATA_PATH, [SMILES_COL, *KEEP_COLS], optional=[LABEL_COL] if LABEL_COL else [])
    rows.insert(0, "row_index", range(len(rows)))  # row in the input file
    assert emb.index.max() < len(rows), "Embedding row_index does not fit the CSV"
    mismatch = (rows.loc[emb.index, SMILES_COL].astype(str).str.strip().values != emb["smiles"].values).sum()
    assert mismatch == 0, f"{mismatch} rows have different SMILES in the embeddings and {Path(DATA_PATH).name}"
else:
    rows = emb[["smiles"]].reset_index()  # row_index, smiles

HAS_LABELS = (DATA_PATH is not None and LABEL_COL is not None and LABEL_COL in rows.columns
              and rows[LABEL_COL].notna().any())
print(f"{len(rows):,} rows, {len(emb):,} with {VIEW} embeddings ({X.shape[1]} dims)")

# Test molecules that were also in the LightGBM training CSV make the metrics look better than they are
train_csv = Path(lgbm_config["data_path"])
if train_csv.exists():
    train_smiles = set(read_table(train_csv, [lgbm_config["smiles_col"]]).iloc[:, 0].astype(str).str.strip())
    overlap = int(emb["smiles"].isin(train_smiles).sum())
    if overlap:
        print(f"WARNING: {overlap:,} of {len(emb):,} test molecules are also in the training CSV ({train_csv.name}); "
              "metrics on them are optimistic.")
if len(emb) < len(rows):
    print(f"{len(rows) - len(emb):,} rows have no embedding (invalid SMILES) and get empty scores")
print(f"Labels: {'yes' if HAS_LABELS else 'no (metrics will be skipped)'}")

1,031 rows, 1,031 with fused embeddings (512 dims)
Labels: yes


## 3. Score with every fold model

In [5]:
import lightgbm as lgb

predictions = rows.copy()
has_emb = predictions.index.isin(emb.index) if DATA_PATH is not None else np.ones(len(rows), bool)
order = emb.index if DATA_PATH is not None else predictions.index  # rows that X lines up with

fold_cols = []
for i, path in enumerate(MODEL_PATHS):
    booster = lgb.Booster(model_file=str(path))
    assert booster.num_feature() == X.shape[1], (
        f"{path.name} expects {booster.num_feature()} features, embeddings have {X.shape[1]}"
    )
    col = f"score_fold{i}"
    predictions[col] = np.nan
    predictions.loc[order, col] = booster.predict(X)
    fold_cols.append(col)

predictions["score"] = predictions[fold_cols].mean(axis=1)  # ensemble: mean over folds
predictions["pred"] = (predictions["score"] > THRESHOLD).astype("Int64").where(has_emb)
predictions["rank"] = predictions["score"].rank(ascending=False, method="first").astype("Int64")
predictions.to_csv(OUT_DIR / "predictions.csv", index=False)

print(f"Saved {len(predictions):,} rows to {OUT_DIR / 'predictions.csv'}")
display(predictions.sort_values("rank").head(10))

Saved 1,031 rows to ~/repos/biomed-multi-view/aircheck/output/sample-data/lgbm/lgbm-fused-finetuned/predictions/test/predictions.csv


,row_index,SMILES,LABEL,score_fold0,score,pred,rank
284,284,CN1C(=O)C(c2c(-c3ccc(Cl)cc3)[nH]c3c2c(=O)[nH]c...,1,0.932028,0.932028,1,1
1029,1029,CCn1c(SCc2conc2C)nnc1-c1cccc(Cl)c1,0,0.862601,0.862601,1,2
891,891,C[C@@H](NC(=O)c1ccc2c(c1)CCC(=O)N2)c1ccc(Br)cc1,1,0.634814,0.634814,1,3
869,869,CC(c1cccc(Cl)c1F)N1CCC(c2nc(C(N)=O)c[nH]2)CC1,0,0.350749,0.350749,0,4
828,828,N#Cc1cnn(-c2ccc(NC(=O)C3(c4ccccc4)CC3)cc2)c1N,0,0.306648,0.306648,0,5
278,278,O=C(c1ccncc1F)N1C[C@H]2CCC[C@@]2(c2nc(CC3CC3)n...,0,0.269469,0.269469,0,6
65,65,O=C(O)c1cc2cc(NC(=O)C3CC34CCS(=O)(=O)CC4)ccc2s1,0,0.254348,0.254348,0,7
313,313,CCNC(=O)C1CCCN(CCn2ccc3cc(Cl)ccc32)C1,0,0.254126,0.254126,0,8
492,492,O=C(Nc1c(-c2ccccc2)nc2ccccn12)c1cccc(F)c1,1,0.234091,0.234091,0,9
521,521,CC1CCC(Oc2cc(CNC(=O)N3CC4CCC3CN4)ccn2)CC1,0,0.217928,0.217928,0,10


## 4. Metrics (if labels are available)

One row per fold model, plus `ensemble` (the mean score).

In [6]:
from sklearn.metrics import average_precision_score, roc_auc_score


def compute_metrics(y, scores) -> dict:
    y, scores = np.asarray(y).astype(int), np.asarray(scores, dtype=float)
    top = np.argsort(-scores, kind="stable")
    metrics = {f"hit@{k}": int(y[top[:k]].sum()) for k in HIT_KS}
    pred = scores > THRESHOLD
    tp, fp = int((pred & (y == 1)).sum()), int((pred & (y == 0)).sum())
    tn, fn = int((~pred & (y == 0)).sum()), int((~pred & (y == 1)).sum())
    both_classes = len(set(y)) == 2
    metrics.update(
        rocauc=roc_auc_score(y, scores) if both_classes else float("nan"),
        pr_auc=average_precision_score(y, scores) if both_classes else float("nan"),
        precision=tp / (tp + fp) if tp + fp else 0.0,
        recall=tp / (tp + fn) if tp + fn else 0.0,
        specificity=tn / (tn + fp) if tn + fp else 0.0,
        accuracy=(tp + tn) / len(y),
    )
    return metrics


if not HAS_LABELS:
    print("No labels; skipping metrics.")
else:
    scored = predictions[has_emb & predictions[LABEL_COL].notna()]
    metrics = pd.DataFrame([
        dict(model=c.removeprefix("score_"), **compute_metrics(scored[LABEL_COL], scored[c]))
        for c in fold_cols + ["score"]
    ]).replace({"model": {"score": "ensemble"}})
    metrics.to_csv(OUT_DIR / "metrics.csv", index=False)
    print(f"{len(scored):,} labeled molecules, {int(scored[LABEL_COL].sum()):,} positives")
    display(metrics.round(4))

1,031 labeled molecules, 31 positives


,model,hit@200,hit@100,hit@50,rocauc,pr_auc,precision,recall,specificity,accuracy
0,fold0,16,10,5,0.6547,0.1366,0.6667,0.0645,0.999,0.9709
1,ensemble,16,10,5,0.6547,0.1366,0.6667,0.0645,0.999,0.9709


In [7]:
# Files produced
for path in sorted(OUT_DIR.rglob("*")):
    if path.is_file():
        print(f"{str(path.relative_to(OUT_DIR)):30s} {path.stat().st_size / 1e6:8.2f} MB")

config.json                        0.00 MB
metrics.csv                        0.00 MB
predictions.csv                    0.10 MB
